# Module 08 — The Training Loop (notebook)

Walkthrough of the framework files in this directory. We'll:

1. Build a Qwen3 model and inspect its parameter breakdown.
2. Build the AdamW optimizer with the canonical param-group split.
3. Run **one full training step** — forward, backward, clip, step — by hand, on CPU. Verify the loss at init is $\sim \ln(\text{vocab})$ and the loss decreases after one step.
4. Demonstrate gradient accumulation (multiple micro-batches → one optimizer step).
5. Save and reload a checkpoint.
6. Show what `torchrun train.py` does at the command-line level.

**Compute:** CPU is enough.  
**Time:** ~10 minutes.

**Note:** FSDP2 wrapping requires a torch.distributed process group. We skip the FSDP wrap in this notebook (it would no-op on single-process anyway) and demonstrate it via the `train.py` command-line below.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from config import ModelConfig, OptimizerConfig, DataConfig, TrainingConfig
from model import build_model, count_params
from optim import build_optimizer
from data import make_dataloader, cycle
from loop import forward_loss, train_step

torch.manual_seed(0)
device = "cpu"  # CPU walkthrough; train.py auto-detects CUDA at launch
print("torch:", torch.__version__)

## 1. Build the model — Qwen3 architecture, course-sized

Pick a small config so everything runs in seconds on CPU. The defaults in `config.py` are a ~150M production-shaped model; here we'll go ~3M for the walkthrough.

In [ ]:
model_cfg = ModelConfig(
    vocab_size=2048,
    d_model=128,
    n_layers=2,
    n_heads=4,
    n_kv_heads=2,
    d_ffn=256,
    max_seq=128,
    tie_weights=True,
)
model = build_model(model_cfg).to(device)

counts = count_params(model)
for k, v in counts.items():
    print(f"  {k:10s} {v/1e6:>7.3f}M  ({v/counts['total']*100:>5.1f}%)")

## 2. Build the AdamW optimizer with param groups

The decay/no-decay split: weight decay applies to 2D+ parameters; norm `gamma` and biases (1D) are excluded.

In [ ]:
opt_cfg = OptimizerConfig(lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
optimizer = build_optimizer(model, opt_cfg)

for g in optimizer.param_groups:
    n = sum(p.numel() for p in g["params"])
    print(f"  group '{g.get('name','?')}': lr={g['lr']:.2e}  wd={g['weight_decay']}  n_params={n:,}")

Almost all params are in the decay group — the no-decay group is just the RMSNorm `gamma`s (and any biases, which Qwen3 has zero of).

## 3. One training step, by hand

Forward → loss → backward → clip → step. Run this twice and verify the loss decreases.

(We use `dtype='fp32'` because CPU doesn't accelerate BF16 — this is just the walkthrough; production runs use BF16 on GPU.)

In [ ]:
data_cfg = DataConfig(seq_len=64, batch_size_per_device=4, synthetic_samples=200, num_workers=0)
loader = make_dataloader(data_cfg, vocab_size=model_cfg.vocab_size)
batch_iter = cycle(loader)

# Step 1
loss1, gn1 = train_step(model, optimizer, batch_iter,
                        grad_accum=1, grad_clip=1.0, dtype="fp32", device=device)
import math
expected = math.log(model_cfg.vocab_size)
print(f"step 0:  loss={loss1:.4f}  grad_norm={gn1:.3f}  (expected loss-at-init ~{expected:.3f})")
assert abs(loss1 - expected) < 1.0, "loss-at-init off by more than 1 nat — model init bug"

# Step 2 — verify loss decreases
loss2, gn2 = train_step(model, optimizer, batch_iter,
                        grad_accum=1, grad_clip=1.0, dtype="fp32", device=device)
print(f"step 1:  loss={loss2:.4f}  grad_norm={gn2:.3f}")
assert loss2 < loss1, "loss didn't decrease — check LR and optimizer setup"
print("\nLoss decreased — training loop is working.")

Two things to note:

- **Loss at init = $\sim \ln(\text{vocab})$.** This is the cheapest sanity test we have. If it's off by more than ~1 nat, the model init is broken — re-read Module 07 § 8 before launching anything bigger.
- **Grad norm is small (<1).** Healthy. If it's > 10 at init, your LR is too high or there's an init scale problem.

## 4. Gradient accumulation

Set `grad_accum=4` and watch the same loop accumulate gradients over 4 micro-batches per optimizer step. Loss reported by `train_step` is the mean over the 4 micro-batches; the model takes one update.

In [ ]:
# Reset model + optimizer for a clean comparison.
torch.manual_seed(0)
model2 = build_model(model_cfg).to(device)
optimizer2 = build_optimizer(model2, opt_cfg)
batch_iter2 = cycle(make_dataloader(data_cfg, vocab_size=model_cfg.vocab_size))

print("With grad_accum=4 (4 micro-batches per optimizer step):")
for step in range(3):
    loss, gn = train_step(model2, optimizer2, batch_iter2,
                          grad_accum=4, grad_clip=1.0, dtype="fp32", device=device)
    print(f"  step {step}:  loss={loss:.4f}  grad_norm={gn:.3f}")

The reported loss should be very close to what you'd get from a 4× larger batch in one step — that's the whole point of accumulation. The trick is that each micro-batch's loss is divided by `grad_accum` *before* `loss.backward()`, so the accumulated `.grad` is the average gradient (not the sum).

On a real run, `effective_batch_size = batch_size_per_device × world_size × grad_accum`. You choose `grad_accum` to hit a target effective batch (typically 1M–4M tokens per step for LLM pretraining) given how much fits per device.

## 5. Checkpointing

Save the current model + optimizer state, then load it into a fresh model and verify the parameters match.

In [ ]:
from checkpoint import save, load
import tempfile, os, shutil

tmp = tempfile.mkdtemp(prefix="ckpt_test_")
ckpt_path = save(model, optimizer, step=42, out_dir=tmp)
print(f"saved to: {ckpt_path}")
print(f"contents: {os.listdir(ckpt_path)}")

# Build a fresh model + optimizer; load into it; compare weights.
torch.manual_seed(99)  # different seed so init is different from saved model
fresh_model = build_model(model_cfg).to(device)
fresh_opt = build_optimizer(fresh_model, opt_cfg)

# Sanity-check that weights are different BEFORE loading.
p_orig = next(model.parameters())
p_fresh = next(fresh_model.parameters())
diff_before = (p_orig - p_fresh).abs().max().item()

loaded_step = load(fresh_model, fresh_opt, ckpt_path)
diff_after = (p_orig - next(fresh_model.parameters())).abs().max().item()

print(f"\nstep loaded:        {loaded_step}")
print(f"weight diff BEFORE: {diff_before:.4f}")
print(f"weight diff AFTER:  {diff_after:.6e}  (should be ~0)")
assert diff_after < 1e-6, "checkpoint reload didn't restore weights"

shutil.rmtree(tmp)

Checkpoint reload restores weights to bit-for-bit equality (well, modulo non-deterministic stream order; here we're within rounding). In distributed training, `dcp.save` writes per-rank shards into the checkpoint directory and `dcp.load` rehydrates them into whatever sharding the current model has — including a different number of GPUs.

## 6. Launching `train.py` via torchrun

The notebook is for understanding the pieces; real runs go through `torchrun`. The minimum command:

```bash
torchrun --standalone --nproc_per_node=1 train.py \
    --total_steps=20 --log_every=5 \
    --vocab_size=2048 --d_model=128 --n_layers=2 \
    --batch_size_per_device=2
```

This runs the same loop you just stepped through, but launched the way every script in Part 3 launches:

- `torchrun` populates `RANK`, `LOCAL_RANK`, `WORLD_SIZE` environment variables.
- `train.py` calls `init_distributed()`, sets the CUDA device, creates the NCCL process group.
- The model is built and wrapped with FSDP2 (`fully_shard`).
- The loop runs, logging every `log_every` steps and saving every `save_every`.

**To scale to 8 GPUs**, change one flag: `--nproc_per_node=8`. The training script is unchanged. FSDP shards the model across the 8 ranks; communication is hidden inside the FSDP wrappers.

**To scale to multi-node**, set `--nnodes`, `--rdzv_backend`, `--rdzv_endpoint`. Still the same script.

Module 11 ships a full demo config (`configs/demo.yaml`) and a launch script that targets the FineWeb-Edu data pipeline. The work you did in this notebook is what runs *inside* every step of that launch.

## Recap

You now have:

- A working Qwen3 model builder (swap-friendly to Llama, DeepSeek, or your Part-2 model).
- An AdamW optimizer with the LLM-canonical param-group split (weight decay only on 2D+ params).
- A `train_step` that handles gradient accumulation, BF16 mixed precision, FSDP-aware `no_sync`, and gradient clipping.
- DCP-based checkpoint save/load that scales to FSDP2-sharded training.
- A `torchrun`-launchable `train.py` that's the same code on 1 GPU or 64 GPUs.

**Next:** [Module 09 — The Learning Rate](../09-learning-rate/). The most important hyperparameter in training. Warmup, cosine decay, muP transfer. Adds `schedule.py` to the framework and a one-line scheduler step to `train.py`.